# 1. Import dataset

## 1.1 Import safe/ vulnerable dataset and attach labels, in particular
+ Safe package -> 0
+ vulnerable package -> 1

In [223]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # or project root

In [224]:
import pandas as pd
import random

In [225]:
df_safe = pd.read_csv("../data/safe_packages.txt")
df_safe["isRisky"] = 0
print(df_safe.head(5))
print(f"Imported safe packages into dataframe with shape {df_safe.shape}")


df_vulnerable = pd.read_csv("../data/vulnerable_packages.txt").sample(n=5000)
df_vulnerable["isRisky"] = 1
print(df_vulnerable.head(5))
print(f"Imported vulnerable packages into dataframe with shape {df_vulnerable.shape}")

df = pd.concat([df_safe, df_vulnerable], ignore_index=True)
print(df.head(5))
print(f"Shape of combined dataframe: {df.shape}")

X = df.drop(['isRisky'], axis=1)
y = df['isRisky']

                                package_name@version  isRisky
0                                       komlib@0.1.0        0
1                          juegodeguerrajavier@7.1.2        0
2  odoo12-addon-mail-outbound-static@12.0.1.0.1.9...        0
3                                   momba@1.0.0.dev4        0
4                                    homestock@1.6.8        0
Imported safe packages into dataframe with shape (4786, 2)
                           package_name@version  isRisky
24043  opencv-contrib-python-headless@3.4.17.63        1
29386                   pytorch-lightning@0.7.1        1
40415                             weblate@3.9.1        1
15969                      knowledge-repo@0.6.2        1
19638                  matrix-synapse@1.85.0rc2        1
Imported vulnerable packages into dataframe with shape (5000, 2)
                                package_name@version  isRisky
0                                       komlib@0.1.0        0
1                          juegodeguer

## 1.2 Split training and testing set

In [226]:
from sklearn.model_selection import train_test_split

In [227]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y,random_state=42)
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (7828, 1)
Shape of X_test: (1958, 1)
Shape of y_train: (7828,)
Shape of y_test: (1958,)


## 2 Training Loop - Test: using one iteration

In [228]:
from scripts.GraphGenerator import GraphGenerator
from scripts.FeatureGenerator import FeatureGenerator

In [229]:
feature_generator_pypi = FeatureGenerator('pypi')

In [230]:
# Get a package@version from X_train 
pkg = X_train['package_name@version'].values[0]
package_name, version = pkg.rsplit('@', 1)
graph_generator = GraphGenerator(package_name, version)


[GraphGenerator] Initializing 1st layer of the graph...
[GraphGenerator] Fetching dependencies...
Level: 3
[['strawberry-graphql@0.93.13', 'cached-property@1.5.2', 'click@8.3.1', 'graphql-core@3.1.7', 'pygments@2.20.0', 'python-dateutil@2.9.0.post0', 'python-multipart@0.0.5', 'sentinel@0.3.0', 'six@1.17.0', 'typing-extensions@4.15.0']]
[GraphGenerator] Fetching dependencies...
[GraphGenerator] Fetching dependencies...
[GraphGenerator] Fetching dependencies...
[GraphGenerator] Fetching dependencies...
[GraphGenerator] Fetching dependencies...
[GraphGenerator] Fetching dependencies...
[GraphGenerator] Fetching dependencies...
[GraphGenerator] Fetching dependencies...
[GraphGenerator] Fetching dependencies...
Level: 2
[['strawberry-graphql@0.93.13', 'cached-property@1.5.2', 'click@8.3.1', 'graphql-core@3.1.7', 'pygments@2.20.0', 'python-dateutil@2.9.0.post0', 'python-multipart@0.0.5', 'sentinel@0.3.0', 'six@1.17.0', 'typing-extensions@4.15.0'], []]


In [231]:
import networkx as nx
from pyvis.network import Network

In [232]:
G = nx.DiGraph()

for package_id, node in graph_generator.nodes_map.items():
    G.add_node(package_id)
    for dep_id in node.depends_on:
        G.add_edge(package_id, dep_id)

net = Network(height="900px", width="100%", directed=True, notebook=False)
net.from_nx(G)
net.write_html("dependency_graph.html", notebook=False)

In [233]:
import importlib, inspect
import scripts.FeatureGenerator as fg_mod
print(fg_mod.__file__)
importlib.reload(fg_mod)
from scripts.FeatureGenerator import FeatureGenerator
feature_generator_pypi = FeatureGenerator("pypi")


/Users/xipingye/software-supply-chain-risk-detector/scripts/FeatureGenerator.py


In [234]:
for p in graph_generator.nodes_map:
    res = feature_generator_pypi.get_full_features(p, graph_generator.nodes_map)
    graph_generator.nodes_map[p].features = res['full_metadata']
    print(res['full_metadata'])
    print(res['col_names'])

[1, 0, False, False, True, True, True, False, 2, 2, 0, 8]
['num_authors', 'num_maintainers', 'has_license', 'yanked', 'has_project_url', 'has_package_url', 'has_release_url', 'has_organization', 'num_roles', 'num_distributions', 'in_degree', 'out_degree']
[1, 0, False, False, True, True, True, False, 1, 2, 1, 0]
['num_authors', 'num_maintainers', 'has_license', 'yanked', 'has_project_url', 'has_package_url', 'has_release_url', 'has_organization', 'num_roles', 'num_distributions', 'in_degree', 'out_degree']
[0, 1, True, False, True, True, True, True, 0, 2, 1, 0]
['num_authors', 'num_maintainers', 'has_license', 'yanked', 'has_project_url', 'has_package_url', 'has_release_url', 'has_organization', 'num_roles', 'num_distributions', 'in_degree', 'out_degree']
[1, 0, False, False, True, True, True, False, 3, 2, 1, 0]
['num_authors', 'num_maintainers', 'has_license', 'yanked', 'has_project_url', 'has_package_url', 'has_release_url', 'has_organization', 'num_roles', 'num_distributions', 'in_d

In [235]:
import torch

In [236]:
node_ids = list(graph_generator.nodes_map.keys())
node_to_idx = {nid: i for i, nid in enumerate(node_ids)}

x_tensor = torch.tensor(
    [graph_generator.nodes_map[nid].features for nid in node_ids],
    dtype=torch.float
)

print(f"Shape of tensor: {x_tensor.shape}")

Shape of tensor: torch.Size([10, 12])


In [237]:
edges = []

for nid, node in graph_generator.nodes_map.items():
    src = node_to_idx[nid]
    for dep in node.depends_on:
        if dep in node_to_idx:  # ensure inside subgraph
            dst = node_to_idx[dep]
            edges.append([src, dst])

edge_index = torch.tensor(edges, dtype=torch.long).t()

In [238]:
print(edges)

[[0, 5], [0, 6], [0, 2], [0, 4], [0, 9], [0, 3], [0, 7], [0, 1], [5, 8]]


In [239]:
edge_index = torch.cat([edge_index, edge_index[[1, 0]]], dim=1)

In [240]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

In [241]:
class GNN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = nn.Linear(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        return x  # node embeddings

In [249]:
label = y_train.values[0]
target_node_idx = 0 

In [250]:
model = GNN(in_dim=x_tensor.shape[1], hidden_dim=64, out_dim=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()

# Forward
node_embeddings = model(x_tensor, edge_index)

# Only use ROOT node
target_embedding = node_embeddings[target_node_idx]

# Classification
logits = model.lin(target_embedding)

# Loss
label_tensor = torch.tensor([label], dtype=torch.long)
loss = F.cross_entropy(logits.unsqueeze(0), label_tensor)

# Backward
optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())

Loss: 0.7650208473205566
